# Démonstration du Backend Dask

Ce notebook montre comment utiliser le backend Dask pour exécuter une simulation Seapopym de manière distribuée (ou parallélisée en local).

Nous allons créer un modèle simple avec des tâches artificiellement lourdes pour illustrer l'intérêt du parallélisme.

In [ ]:
import time
import xarray as xr
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import dask
from dask.distributed import Client

from seapopym.controller import SimulationController, SimulationConfig
from seapopym.blueprint import Blueprint
from seapopym.standard import Coordinates

## 1. Configuration du Cluster Dask Local

Nous démarrons un `Client` Dask. Cela va créer un cluster local (scheduler + workers) sur votre machine. Vous pourrez voir le tableau de bord Dask (lien affiché ci-dessous) pour suivre l'exécution.

In [ ]:
# Démarrage du cluster local
client = Client(
    n_workers=2,
    threads_per_worker=2,
    memory_limit="16GB",
)
client

## 2. Définition du Modèle

Nous définissons des fonctions de calcul qui simulent une charge de travail (avec `time.sleep`).

In [ ]:
def heavy_computation(x):
    """Simule un calcul lourd sur une grille."""
    # En réalité, avec Dask, on manipulerait des dask arrays.
    # Ici on simule juste le temps de traitement de la tâche elle-même.
    return {"result_a": x * 2}

def another_heavy_computation(x):
    """Un autre calcul indépendant."""
    return {"result_b": x + 10}

def combine_results(a, b):
    """Combine les résultats précédents."""
    return {"final": a + b}

def configure_model(bp: Blueprint):
    bp.register_forcing("input_data")
    
    # Groupe A
    bp.register_group(
        "GroupA",
        [{
            "func": heavy_computation,
            "output_mapping": {"result_a": "res_a"},
            "input_mapping": {"x": "input_data"}
        }]
    )
    
    # Groupe B (Indépendant de A)
    bp.register_group(
        "GroupB",
        [{
            "func": another_heavy_computation,
            "output_mapping": {"result_b": "res_b"},
            "input_mapping": {"x": "input_data"}
        }]
    )
    
    # Groupe C (Dépend de A et B)
    bp.register_group(
        "GroupC",
        [{
            "func": combine_results,
            "output_mapping": {"final": "final_res"},
            "input_mapping": {"a": "GroupA/res_a", "b": "GroupB/res_b"}
        }]
    )

## 3. Configuration de la Simulation

In [ ]:
start = datetime(2020, 1, 1)
end = datetime(2022, 1, 1) # 4 jours
config = SimulationConfig(start_date=start, end_date=end, timestep=timedelta(days=1))

# Création du Controller avec le backend Dask
controller = SimulationController(config, backend="dask")

## 4. Données d'Entrée (Forçages)

Nous créons un dataset de forçage.

In [ ]:
size = 1000
times = pd.date_range(start, end, freq="D")
data = np.random.rand(len(times), size, size)

forcings = xr.Dataset(
    {"input_data": ((Coordinates.T, "x", "y"), data)},
    coords={Coordinates.T: times, "x": range(size), "y": range(size)}
)

# État initial (vide ici car tout vient des forçages)
initial_state = xr.Dataset(coords={"x": range(size), "y": range(size)})

## 5. Exécution

Nous lançons la simulation. Observez le dashboard Dask pendant l'exécution !

In [ ]:
print("Initialisation...")
controller.setup(configure_model, initial_state, forcings=forcings)

print("Démarrage de la simulation...")
start_time = time.time()

controller.run()

end_time = time.time()
print(f"Simulation terminée en {end_time - start_time:.2f} secondes.")

## 6. Analyse des Résultats

In [ ]:
print("Variables produites :", list(controller.state.data_vars))
print("\nExemple de résultat (GroupC/final_res) :")
print(controller.state["GroupC/final_res"].isel(x=0, y=0).values)

In [ ]:
# N'oubliez pas de fermer le client
client.close()